<a href="https://colab.research.google.com/github/wbutterfliez/NLP/blob/main/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install
!pip install tensorflow
!pip install transformers datasets torch

# Imports
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from transformers import pipeline

In [ ]:
vocab_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

word_index = imdb.get_word_index()

x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

model = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train, epochs=2, batch_size=64)

def encode_text(text):
    tokens = text.lower().split()
    encoded = [word_index.get(word, 2) for word in tokens]  # 2 = unknown
    return pad_sequences([encoded], maxlen=max_len)

sample = "this movie was amazing and beautiful"
encoded = encode_text(sample)

prediction = model.predict(encoded)[0][0]

print(sample)
print("Sentiment:", "Positive" if prediction > 0.5 else "Negative")

In [ ]:
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train, epochs=2, batch_size=64)

def encode_text(text):
    tokens = text.lower().split()
    encoded = [word_index.get(word, 2) for word in tokens]
    return pad_sequences([encoded], maxlen=max_len)

sample = "this movie was terrible and boring"
encoded = encode_text(sample)

prediction = model.predict(encoded)[0][0]

print(sample)
print("Sentiment:", "Negative" if prediction > 0.5 else "Positive")

In [ ]:
classifier = pipeline("sentiment-analysis")

texts = [
    "I love this movie!",
    "This was the worst thing ever",
    "Honestly kinda mid"
]

results = classifier(texts)

for text, res in zip(texts, results):
    print(text, "->", res)

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch
import torch.nn.functional as F


model_name = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name)

model.eval()

labels = ["Negative", "Neutral", "Positive"]

def predict_sentiment(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)


    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits


    probs = F.softmax(logits, dim=1)

    predicted_class = torch.argmax(probs, dim=1).item()

    return labels[predicted_class], probs[0].tolist()

texts = [
    "this movie was amazing and beautiful",
    "this movie was terrible and boring",
    "it was okay, not great",
    "average experience nothing special"
]

for text in texts:
    label, prob = predict_sentiment(text)

    print("Text:", text)
    print("Sentiment:", label)
    print("Probabilities [Neg, Neu, Pos]:", prob)
    print()